In [1]:


from pathlib import Path
import os
import json
import time

import pandas as pd
from tqdm.auto import tqdm

from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env
load_dotenv()

# Create OpenAI client 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Project paths
PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data dir:", DATA_PROCESSED)


c:\Users\User\Documents\retailmind-prototype\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: C:\Users\User\Documents\retailmind-prototype
Processed data dir: C:\Users\User\Documents\retailmind-prototype\data\processed


In [2]:
# Load enriched English turns (SGD + CCPE until now)
df = pd.read_parquet(DATA_PROCESSED / "uss_english_turns_satisfaction.parquet")

print("Total rows:", len(df))
df["speaker"].value_counts()
#df.head()


Total rows: 39102


speaker
USER      20693
SYSTEM    18409
Name: count, dtype: int64

In [3]:
# Keep only turn-level, low-satisfaction USER turns
df_low = df[
    (df["low_satisfaction"] == True) &
    (df["speaker"] == "USER") &
    (df["is_overall"] == False)
].copy()

print("Low-satisfaction USER turns:", len(df_low))
df_low.head()


Low-satisfaction USER turns: 4089


,conv_id,turn_id,speaker,text,dialog_act,scores_raw,scores,is_overall,satisfaction_mean,label,dataset,low_satisfaction,entity_tag
10,0,11,USER,"No, play it on bedroom speaker instead.",INFORM,"3,2,3","[3, 2, 3]",False,2.666667,INFORM,SGD,True,None
27,1,9,USER,No. Don't do that.,NEGATE_INTENT,"3,3,2,3","[3, 3, 2, 3]",False,2.750000,NEGATE_INTENT,SGD,True,None
29,1,11,USER,Make a reservation at a nearby restaurant.,INFORM_INTENT,"3,3,2,3","[3, 3, 2, 3]",False,2.750000,INFORM_INTENT,SGD,True,None
35,1,17,USER,"No, I need it for 3 at 11:45 am.",INFORM,"3,3,2,3","[3, 3, 2, 3]",False,2.750000,INFORM,SGD,True,None
52,2,7,USER,Can you check on another? I'm looking for some...,INFORM,"3,2,2,3","[3, 2, 2, 3]",False,2.500000,INFORM,SGD,True,None


In [4]:
'''MAX_EXAMPLES = 400  # to limit cost and time for now
if len(df_low) > MAX_EXAMPLES:
    df_low = df_low.sample(MAX_EXAMPLES, random_state=42).copy()
    print("Sampled low-satisfaction turns:", len(df_low))
'''

'MAX_EXAMPLES = 400  # to limit cost and time for now\nif len(df_low) > MAX_EXAMPLES:\n    df_low = df_low.sample(MAX_EXAMPLES, random_state=42).copy()\n    print("Sampled low-satisfaction turns:", len(df_low))\n'

We reconstruct the last SYSTEM turn in each conversation, then attach it to each low-satisfaction USER turn.

In [5]:
# Sort full DF to reconstruct conversation order
df_sorted = df.sort_values(["dataset", "conv_id", "turn_id"]).copy()

# Keep system-only text
df_sorted["system_text_only"] = df_sorted["text"].where(df_sorted["speaker"] == "SYSTEM")


# Propagate the last system turn within each conversation
df_sorted["last_system_text"] = (
    df_sorted
    .groupby(["dataset", "conv_id"])["system_text_only"]
    .ffill()
)

df_sorted[["dataset", "conv_id", "turn_id", "speaker", "text", "last_system_text"]].head()


,dataset,conv_id,turn_id,speaker,text,last_system_text
26666,CCPE,0,1,SYSTEM,Do you like movies like Thor?,Do you like movies like Thor?
26667,CCPE,0,2,USER,"No, I don't like Thor.",Do you like movies like Thor?
26668,CCPE,0,3,SYSTEM,Ok. What is it about this type of movie that y...,Ok. What is it about this type of movie that y...
26669,CCPE,0,4,USER,I don't like all the,Ok. What is it about this type of movie that y...
26670,CCPE,0,5,USER,Like the I don't know. Like is it voice acting?,Ok. What is it about this type of movie that y...


In [6]:
df_sorted["last_system_text"].isnull().sum()

np.int64(1010)

In [7]:
# Merge last_system_text onto low-satisfaction USER turns
df_low = df_low.merge(
    df_sorted[["dataset", "conv_id", "turn_id", "last_system_text"]],
    on=["dataset", "conv_id", "turn_id"],
    how="left"
)

df_low["last_system_text"] = df_low["last_system_text"].fillna(
    "[No previous system turn available]"
)


# Build snippet: what the bot said + user's unhappy message
df_low["snippet"] = (
    "Bot: " + df_low["last_system_text"].astype(str) + "\n"
    "User: " + df_low["text"].astype(str)
)

df_low[["dataset", "conv_id", "turn_id", "snippet"]].head()


,dataset,conv_id,turn_id,snippet
0,SGD,0,11,Bot: Play Mamma knows best on the kitchen spea...
1,SGD,1,9,Bot: So should I pick you up some tickets?\nUs...
2,SGD,1,11,Bot: What should I do?\nUser: Make a reservati...
3,SGD,1,17,Bot: Table for 2 at McDonald's in San Ramon on...
4,SGD,2,7,Bot: 2033 Camden Avenue # F3.\nUser: Can you c...


In [8]:
# Taxonomy of issue labels
ISSUE_LABELS = [
    "MISSING_CONTEXT",
    "WRONG_FACT",
    "TONE_ISSUE",
    "LOOP",
    "UNSUPPORTED_INTENT",
    "SLOW_RESPONSE",
    "HANDOFF_REQUIRED",
    "SUCCESS_BEST_PRACTICE",
]

def build_issue_prompt(snippet: str) -> str:
    """
    Build the prompt given a conversation snippet.
    """
    taxonomy_str = ", ".join(ISSUE_LABELS)
    return f"""
You are an assistant that diagnoses why a chatbot reply led to low user satisfaction.

You receive a short conversation snippet (last bot message and user follow-up).
Your job:
1. Decide which issue categories apply, using this taxonomy:
   {taxonomy_str}
2. Explain briefly why (1-3 sentences).
3. Estimate severity: LOW, MEDIUM, or HIGH.

Guidelines:
- Use MISSING_CONTEXT when the bot ignores prior information or asks for data it already has.
- Use WRONG_FACT when the bot gives incorrect or misleading information.
- Use TONE_ISSUE when the bot is rude, robotic, or lacks empathy while the user is upset.
- Use LOOP when the bot repeats itself or gets stuck.
- Use UNSUPPORTED_INTENT when the user asks for something outside the bot's capabilities.
- Use SLOW_RESPONSE when the delay/latency clearly bothers the user.
- Use HANDOFF_REQUIRED when a human agent should take over.
- If there is no real failure, use SUCCESS_BEST_PRACTICE as the only issue.

Output a JSON object with exactly this schema:
{{
  "issues": ["ISSUE_1", "ISSUE_2", ...],  # at least one issue; use "SUCCESS_BEST_PRACTICE" if no real failure
  "severity": "LOW" | "MEDIUM" | "HIGH",
  "reason": "short explanation here"
}}

Conversation snippet:
---
{snippet}
---
"""


In [9]:
MODEL_NAME = "gpt-4o-mini"  #  try models later
TEMPERATURE = 0.1

def call_issue_tagger(snippet: str, max_retries: int = 3, sleep_seconds: float = 2.0) -> dict:
    """
    Call OpenAI to tag issues for the given snippet.
    Returns a dict: {"issues": [...], "severity": "...", "reason": "..."}.
    """
    prompt = build_issue_prompt(snippet)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                temperature=TEMPERATURE,
                response_format={"type": "json_object"},
                messages=[
                    {
                        "role": "system",
                        "content": "You are an AI assistant specialized in diagnosing chatbot failures. Always output valid JSON."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    },
                ],
            )

            content = response.choices[0].message.content
            data = json.loads(content)

            # Defaults
            data.setdefault("issues", [])
            data.setdefault("severity", "MEDIUM")
            data.setdefault("reason", "")

            # Normalize issues to the allowed labels
            normalized_issues = []
            for issue in data.get("issues", []):
                issue = str(issue).strip().upper()
                if issue in ISSUE_LABELS:
                    normalized_issues.append(issue)

            # If LLM didn't pick any valid label, fall back
            if not normalized_issues:
                normalized_issues = ["SUCCESS_BEST_PRACTICE"]

            data["issues"] = normalized_issues

            # Normalize severity
            sev = str(data.get("severity", "MEDIUM")).upper()
            if sev not in {"LOW", "MEDIUM", "HIGH"}:
                sev = "MEDIUM"
            data["severity"] = sev

            # Reason as string
            data["reason"] = str(data.get("reason", "")).strip()

            return data

        except Exception as e:
            print(f"Error on attempt {attempt + 1}: {e}")
            time.sleep(sleep_seconds)

    # Fallback if all retries fail
    return {
        "issues": ["HANDOFF_REQUIRED"],
        "severity": "HIGH",
        "reason": "Tagging failed due to API error.",
    }


In [10]:
issue_results = []

# iterrows with tqdm
for idx, row in tqdm(df_low.iterrows(), total=len(df_low)):
    snippet = row["snippet"]
    result = call_issue_tagger(snippet)

    issue_results.append({
        "dataset": row["dataset"],
        "conv_id": row["conv_id"],
        "turn_id": row["turn_id"],
        "issues": result["issues"],
        "severity": result["severity"],
        "reason": result["reason"],
    })

len(issue_results)


100%|██████████| 4089/4089 [2:02:34<00:00,  1.80s/it]  


4089

In [11]:
df_issues = pd.DataFrame(issue_results)
df_issues.head()


,dataset,conv_id,turn_id,issues,severity,reason
0,SGD,0,11,[MISSING_CONTEXT],MEDIUM,The bot fails to acknowledge the user's reques...
1,SGD,1,9,[MISSING_CONTEXT],MEDIUM,The bot fails to acknowledge the user's clear ...
2,SGD,1,11,[UNSUPPORTED_INTENT],HIGH,The bot is unable to fulfill the user's reques...
3,SGD,1,17,[MISSING_CONTEXT],MEDIUM,The bot failed to acknowledge the user's previ...
4,SGD,2,7,[MISSING_CONTEXT],MEDIUM,The bot fails to acknowledge the user's previo...


In [12]:
# Merge with the full dataset (all turns, not only low-satisfaction ones)
df_tagged = df.merge(
    df_issues,
    on=["dataset", "conv_id", "turn_id"],
    how="left",
)

# For rows that were not tagged (non low-satisfaction), fill defaults
df_tagged["issues"] = df_tagged["issues"].apply(
    lambda x: x if isinstance(x, list) else []
)
df_tagged["severity"] = df_tagged["severity"].fillna("NONE")
df_tagged["reason"] = df_tagged["reason"].fillna("")

print("Rows in tagged DF:", len(df_tagged))
df_tagged.head()


Rows in tagged DF: 39102


,conv_id,turn_id,speaker,text,dialog_act,scores_raw,scores,is_overall,satisfaction_mean,label,dataset,low_satisfaction,entity_tag,issues,severity,reason
0,0,1,USER,What is the weather like on the March 4th?,INFORM,"3,3,3","[3, 3, 3]",False,3.000000,INFORM,SGD,False,None,[],NONE,
1,0,2,SYSTEM,In which city should I look?,REQUEST,None,None,False,NaN,REQUEST,SGD,False,None,[],NONE,
2,0,3,USER,The weather in Mill Valley.,INFORM,"4,3,3","[4, 3, 3]",False,3.333333,INFORM,SGD,False,None,[],NONE,
3,0,4,SYSTEM,The weather should be around 90 degrees and th...,OFFER,None,None,False,NaN,OFFER,SGD,False,None,[],NONE,
4,0,5,USER,How humid will the temperature be?,REQUEST,"3,3,3","[3, 3, 3]",False,3.000000,REQUEST,SGD,False,None,[],NONE,


In [13]:
# Save final issue-tagged dataset
out_path = DATA_PROCESSED / "uss_english_issue_tagged.parquet"
df_tagged.to_parquet(out_path, index=False)
print("Saved issue-tagged dataset to:", out_path)


Saved issue-tagged dataset to: C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_issue_tagged.parquet


In [14]:
from collections import Counter

issue_counter = Counter()
for issues in df_tagged["issues"]:
    if isinstance(issues, list):
        for issue in issues:
            issue_counter[issue] += 1

print("Issue counts:", issue_counter)

print("\nSeverity counts:")
print(df_tagged["severity"].value_counts())


Issue counts: Counter({'MISSING_CONTEXT': 3320, 'SUCCESS_BEST_PRACTICE': 552, 'UNSUPPORTED_INTENT': 187, 'TONE_ISSUE': 67, 'WRONG_FACT': 12, 'HANDOFF_REQUIRED': 9, 'LOOP': 4, 'SLOW_RESPONSE': 1})

Severity counts:
severity
NONE      35013
MEDIUM     3390
LOW         553
HIGH        146
Name: count, dtype: int64


In [15]:
sample_issue = "MISSING_CONTEXT"  # change to any label
cols = ["dataset", "conv_id", "turn_id", "speaker", "text", "issues", "severity", "reason"]

df_tagged[df_tagged["issues"].apply(lambda lst: sample_issue in lst)].head(10)[cols]


,dataset,conv_id,turn_id,speaker,text,issues,severity,reason
10,SGD,0,11,USER,"No, play it on bedroom speaker instead.",[MISSING_CONTEXT],MEDIUM,The bot fails to acknowledge the user's reques...
27,SGD,1,9,USER,No. Don't do that.,[MISSING_CONTEXT],MEDIUM,The bot fails to acknowledge the user's clear ...
35,SGD,1,17,USER,"No, I need it for 3 at 11:45 am.",[MISSING_CONTEXT],MEDIUM,The bot failed to acknowledge the user's previ...
52,SGD,2,7,USER,Can you check on another? I'm looking for some...,[MISSING_CONTEXT],MEDIUM,The bot fails to acknowledge the user's previo...
66,SGD,2,21,USER,"No, not just yet.",[MISSING_CONTEXT],MEDIUM,The bot fails to acknowledge the user's respon...
70,SGD,2,25,USER,Evening 7:30 would be good.,[MISSING_CONTEXT],MEDIUM,The bot is asking for the time again despite t...
74,SGD,2,29,USER,Try it again for 5:30 in the evening.,[MISSING_CONTEXT],HIGH,The bot fails to acknowledge the user's previo...
113,SGD,3,33,USER,For 10:30 am for 1 person.,[MISSING_CONTEXT],MEDIUM,The bot is asking for information it already h...
173,SGD,5,29,USER,No I dont want to book anything.,[MISSING_CONTEXT],MEDIUM,The bot fails to acknowledge the user's clear ...
190,SGD,6,7,USER,"No, I want 1 ticket to an event on Wednesday n...",[MISSING_CONTEXT],HIGH,The bot failed to acknowledge the user's previ...


In [16]:
from itertools import chain

all_issues = set(chain.from_iterable(df_tagged["issues"]))
print("Distinct issue labels:", all_issues)

print("Distinct severities:", df_tagged["severity"].unique())


Distinct issue labels: {'SUCCESS_BEST_PRACTICE', 'TONE_ISSUE', 'WRONG_FACT', 'UNSUPPORTED_INTENT', 'LOOP', 'MISSING_CONTEXT', 'HANDOFF_REQUIRED', 'SLOW_RESPONSE'}
Distinct severities: ['NONE' 'MEDIUM' 'HIGH' 'LOW']
